#  Implementação de KNN (K Nearest Neighbors)

In [1]:
import numpy as np
import pandas as pd
import joblib

from knn_scratch import NearestNeighborsCustom

In [2]:
# 1. CARREGAR OS DADOS (Com codificação 'latin-1' para os títulos)
ratings = pd.read_csv(
    "../DATA/ml-1m/ratings.dat",
    sep="::",
    names=["UserID", "MovieID", "Rating", "Timestamp"],
    engine="python"
)

movies = pd.read_csv(
    "../DATA/ml-1m/movies.dat",
    sep="::",
    names=["MovieID", "Title", "Genres"],
    engine="python",
    encoding="latin-1"
)

# Criar dicionário de mapeamento: MovieID -> Nome do Filme
movie_titles = dict(zip(movies["MovieID"], movies["Title"]))

In [3]:

# 2. MONTAR A MATRIZ USUÁRIO-ITEM
user_item_matrix = ratings.pivot_table(
    index="UserID", 
    columns="MovieID", 
    values="Rating"
).fillna(0)



In [4]:
# 3. TREINAR O KNN (FILTRAGEM COLABORATIVA BASEADA EM USUÁRIOS)
X_users = user_item_matrix.values
knn_users = NearestNeighborsCustom(n_neighbors=10, metric='cosine')
knn_users.fit(X_users)

In [ ]:

# 4. FUNÇÃO DE RECOMENDAÇÃO COM EXIBIÇÃO DE NOMES
def recomendar_filmes_para_usuario(user_id, top_n=5):
    if user_id not in user_item_matrix.index:
        print(f"Erro: UserID {user_id} não encontrado na base de dados.")
        return

    # Encontra a posição (linha) do usuário na matriz
    user_row_idx = user_item_matrix.index.get_loc(user_id)
    target_user_vector = X_users[user_row_idx]

    # Busca os 10 usuários mais similares (vizinhos)
    distances, indices = knn_users.kneighbors(target_user_vector)
    
    # Índices dos vizinhos (ignorando o próprio usuário na posição 0)
    neighbor_indices = indices[0][1:]
    neighbor_distances = distances[0][1:]

    # Identificar filmes que o usuário alvo AINDA NÃO assistiu (nota == 0)
    movies_watched_by_target = set(user_item_matrix.columns[target_user_vector > 0])

    # Dicionário para acumular a pontuação/ponderação das notas dos vizinhos
    movie_scores = {}

    for neighbor_idx, dist in zip(neighbor_indices, neighbor_distances):
        # Quanto menor a distância, maior a similaridade
        similarity = 1.0 - dist 
        neighbor_ratings = X_users[neighbor_idx]

        # Percorre apenas as notas dos vizinhos para filmes não vistos pelo alvo
        for col_idx, rating in enumerate(neighbor_ratings):
            movie_id = user_item_matrix.columns[col_idx]
            
            # Recomenda apenas se o vizinho deu nota alta (>= 4.0) e o alvo não viu
            if rating >= 4.0 and movie_id not in movies_watched_by_target:
                if movie_id not in movie_scores:
                    movie_scores[movie_id] = 0.0
                # Pontuação ponderada pela similaridade do vizinho
                movie_scores[movie_id] += rating * similarity

    # Ordena os filmes recomendados pela maior pontuação acumulada
    top_recommended_ids = sorted(movie_scores, key=movie_scores.get, reverse=True)[:top_n]

    # --- PRINT DOS RESULTADOS COM NOMES DOS FILMES ---
    print("=" * 65)
    print(f"RECOMENDAÇÕES PARA O USER_ID {user_id}")
    print("=" * 65)
    
    if not top_recommended_ids:
        print("Nenhuma recomendação forte encontrada para este perfil.")
        return

    for rank, movie_id in enumerate(top_recommended_ids, 1):
        nome_filme = movie_titles.get(movie_id, "Título Desconhecido")
        score = movie_scores[movie_id]
        print(f"{rank}. {nome_filme} (ID: {movie_id}) | Score: {score:.2f}")




In [6]:
# 5. EXECUTAR A RECOMENDAÇÃO
recomendar_filmes_para_usuario(user_id=4, top_n=5)

RECOMENDAÇÕES PARA O USER_ID 4
1. Fugitive, The (1993) (ID: 457) | Score: 12.40
2. Indiana Jones and the Last Crusade (1989) (ID: 1291) | Score: 11.58
3. Godfather, The (1972) (ID: 858) | Score: 11.27
4. Terminator 2: Judgment Day (1991) (ID: 589) | Score: 10.28
5. Braveheart (1995) (ID: 110) | Score: 8.95


In [10]:
# Carregar as avaliações
ratings_cols = ['user_id', 'movie_id', 'rating', 'timestamp']
ratings_df = pd.read_csv(
    '../DATA/ml-1m/ratings.dat',
    sep='::',
    engine='python',
    header=None,
    names=ratings_cols
)
print(f"✅ {len(ratings_df)} avaliações carregadas.")

# Criar Dicionário de Timestamps com tipos NATIVOS do Python (int, int): int
# Isso garante o match no `_formatar_timestamp` da API
timestamps_dict = {
    (int(row.user_id), int(row.movie_id)): int(row.timestamp)
    for row in ratings_df.itertuples(index=False)
}

✅ 1000209 avaliações carregadas.


In [12]:
# 5. SALVAR OS MODELOS E ARTEFATOS PARA A API
print("\nSalvando artefatos do modelo...")

# Modelo KNN customizado treinado
joblib.dump(knn_users, "modelo_knn_users.pkl")

# Matriz Usuário-Item (necessária para recuperar o histórico do usuário)
joblib.dump(user_item_matrix, "user_item_matrix.pkl")

# Mapeamento de IDs para nomes dos filmes
joblib.dump(movie_titles, "movie_titles.pkl")

# TimeStamps Dict
joblib.dump(timestamps_dict,'timestamps_dict.pkl')

print("Todos os artefatos foram salvos com sucesso!")


Salvando artefatos do modelo...
Todos os artefatos foram salvos com sucesso!
